# Bluestock Mutual Fund Analytics Platform
## Day 2: Data Cleaning, Star Schema Design & SQLite Loading
**Team:** SR (Sathwik Ryapaka), AB (Abhiram), AK (Ashmeet Kaur)  
**Date:** 20 November 2026

### Day 2 Objectives:
1. **Data Cleaning:** Clean all 10 datasets (handle dates, duplicates, NAV forward-fills, transaction standardization).
2. **Star Schema Architecture:** Design relational dimensional model (`dim_fund`, `dim_date`, `fact_nav`, `fact_transactions`, `fact_performance`, `fact_aum`).
3. **Database Population:** Load all cleaned datasets into `bluestock_mf.db` using SQLAlchemy.
4. **Analytical SQL Queries:** Execute 10 business intelligence queries against SQLite.
5. **Data Dictionary:** Document full data dictionary.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Environment initialized!")

### 1. Database Connection & Table Verification
Connecting to `bluestock_mf.db` and checking all loaded tables.

In [ ]:
DB_PATH = Path("../bluestock_mf.db")
conn = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query("SELECT name, type FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
print("=== Loaded Database Tables ===")
for _, row in tables_df.iterrows():
    count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {row['name']};", conn)['cnt'][0]
    print(f"{row['name']:<24}: {count:>6,} rows")

### 2. Analytical Query 1: Top 5 Mutual Funds by AUM

In [ ]:
q1 = """
SELECT 
    p.amfi_code,
    f.scheme_name,
    f.fund_house,
    f.category,
    p.aum_crore,
    ROUND(p.expense_ratio_pct, 2) AS expense_ratio_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;
"""
df_q1 = pd.read_sql_query(q1, conn)
df_q1

### 3. Analytical Query 4: Investor Capital Inflow by State

In [ ]:
q4 = """
SELECT 
    state,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount_inr) / 10000000.0, 2) AS total_inflow_crore,
    ROUND(AVG(amount_inr), 2) AS avg_transaction_inr
FROM fact_transactions
GROUP BY state
ORDER BY total_inflow_crore DESC;
"""
df_q4 = pd.read_sql_query(q4, conn)

plt.figure(figsize=(12, 5))
sns.barplot(data=df_q4, x="total_inflow_crore", y="state", palette="mako")
plt.title("Mutual Fund Total Inflows by Indian State (₹ Crore)")
plt.xlabel("Inflows (₹ Crore)")
plt.ylabel("State")
plt.tight_layout()
plt.show()

### 4. Analytical Query 7: City Tier Demographics (T30 vs B30)

In [ ]:
q7 = """
SELECT 
    city_tier,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount_inr) / 10000000.0, 2) AS total_volume_crore,
    ROUND(AVG(amount_inr), 2) AS avg_ticket_size_inr,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM fact_transactions), 2) AS transaction_share_pct
FROM fact_transactions
GROUP BY city_tier;
"""
df_q7 = pd.read_sql_query(q7, conn)
df_q7

### 5. Analytical Query 8: AMC AUM Market Share

In [ ]:
q8 = """
SELECT 
    fund_house,
    aum_crore,
    aum_lakh_crore,
    num_schemes,
    ROUND(aum_crore * 100.0 / (SELECT SUM(aum_crore) FROM fact_aum WHERE date = '2025-12-31'), 2) AS market_share_pct
FROM fact_aum
WHERE date = '2025-12-31'
ORDER BY aum_crore DESC;
"""
df_q8 = pd.read_sql_query(q8, conn)

plt.figure(figsize=(10, 6))
plt.pie(df_q8["market_share_pct"], labels=df_q8["fund_house"], autopct="%1.1f%%", startangle=140, colors=sns.color_palette("Set2"))
plt.title("Mutual Fund AMC Market Share Distribution (Dec 2025)")
plt.tight_layout()
plt.show()